# Step 1.2 — Mass ROI Extraction

This notebook reads the OsiriX XML annotations that ship with INbreast and saves one cropped `.npy` patch per `Name="Mass"` ROI. The crops are organised by benign / malignant label (mapped from BI-RADS) and a per-ROI manifest CSV records the bounding-box coordinates so downstream stages can re-crop from the preprocessed image if they need to.

**Inputs:**

- `../data/raw/inbreast/ALL-IMGS/` — DICOM files (410 total).
- `../data/raw/kaggle_inbreast/AllXML/` — OsiriX plist XML annotation files (343 of 410 DICOMs have an XML; the other 67 are "normal / no annotation").
- `../data/raw/inbreast/INbreast.xls` — BI-RADS metadata (used for the benign / malignant label).

**Outputs** (under `../data/outputs/masses/`):

- `benign/<file_id>_mass<i>.npy` — float32 DICOM crop of each benign mass.
- `malignant/<file_id>_mass<i>.npy` — float32 DICOM crop of each malignant mass.
- `{benign,malignant}/<file_id>_mass<i>.png` — 8-bit per-patch min-max preview, **for human inspection only**; downstream consumers read the `.npy`.
- `mass_index.csv` — one row per `Name="Mass"` ROI with `file_id`, `roi_index`, `label`, bounding-box (`y0, y1, x0, x1`), source `dicom_path`, `xml_path`, and the raw `birads` string.

**What "extract" means here:** for each XML file we parse every ROI with `Name="Mass"`, rasterise its polygon to a binary mask on the full DICOM, compute a tight axis-aligned bounding box around that mask (with a 20-pixel margin), and save the cropped pixel array. The crop is saved at its **native** resolution — Steps 2.1 / 2.2 / 2.4 later re-crop from the **preprocessed** full image (`data/outputs/preprocessed/final/*.npy`) at 224×224, reusing the bounding-box coordinates from `mass_index.csv`.

**BI-RADS → label mapping** (Section 4):

- BI-RADS 1 (normal) → **ignored** (no patch extracted; the image has no mass).
- BI-RADS 2, 3 → **benign**.
- BI-RADS 4 / 4a / 4b / 4c / 5 / 6 → **malignant**.

This uses BI-RADS as a suspicion-level proxy, which is what the reference paper does too.

**Downstream consumers:** Steps 2.1 (WS), 2.2 (CNN), 2.3 (Fusion) and 2.4 (Scattering Covariance) all read `mass_index.csv` to enumerate the dataset. They also apply a defensive one-line filter that drops any all-zero patch — one INbreast annotation (`file_id 50994354`, BI-RADS 4a, view `MG_R_ML_ANON`) has a bounding box that falls entirely outside the breast mask, producing a uniformly-zero crop. The filter lives at the downstream consumers, not here (this notebook saves all 116 crops; downstream sees 115); its root cause and effect are in Section 7.1 below.

**Background patches** (fatty / fibroglandular tissue) are now in `Step 1.3 — Background Patch Sampling.ipynb`.


## 1. Map DICOMs to XML annotation files

Before extracting any mass, I confirm the dataset has the XML annotations the rest of the notebook expects. The XML files come from the Kaggle INbreast 2012 mirror (<https://www.kaggle.com/datasets/tommyngx/inbreast2012>) and are stored under `data/raw/kaggle_inbreast/AllXML/`. The next code cell tabulates which DICOMs have a matching XML and which do not. (For reference: the Razali paper uses a YOLO-style mass detector instead of the dataset's hand-drawn polygons; this project uses the polygons directly because they are the canonical INbreast ground truth.)


**Reproducibility.** The 10-row preview below is seeded with the project-wide `SEED = 34`. Mass-ROI extraction itself iterates over every XML annotation deterministically and does not depend on any seed.

In [1]:
from pathlib import Path
import re
import pandas as pd

SEED = 34  # project-wide seed; fixes which rows the preview table samples

DICOM_DIR = Path("../data/raw/inbreast/ALL-IMGS")
XML_DIR   = Path("../data/raw/kaggle_inbreast/AllXML")  # XML lives under the kaggle_inbreast source folder

def leading_id(stem: str) -> str:
    m = re.match(r"(\d+)", stem)
    return m.group(1) if m else stem

dicoms = sorted(DICOM_DIR.glob("*.dcm"))
xmls   = sorted(XML_DIR.glob("*.xml"))

# map id -> path (keep lists in case of duplicates)
dicom_map = {}
for p in dicoms:
    cid = leading_id(p.stem)
    dicom_map.setdefault(cid, []).append(p)

xml_map = {leading_id(p.stem): p for p in xmls}

rows = []
for cid, dcm_list in dicom_map.items():
    for dcm in dcm_list:
        rows.append({
            "case_id": cid,
            "dicom_name": dcm.name,
            "dicom_path": str(dcm),
            "xml_path": str(xml_map[cid]) if cid in xml_map else None,
            "has_xml": cid in xml_map
        })

link_df = pd.DataFrame(rows)

print("DICOM files:", len(dicoms))
print("XML files:", len(xmls))
print("DICOMs with XML:", int(link_df["has_xml"].sum()))
print("DICOMs missing XML:", int((~link_df["has_xml"]).sum()))

display(link_df.sample(10, random_state=SEED))

# show a few missing examples (useful sanity)
missing = link_df[~link_df["has_xml"]].head(20)
if len(missing):
    print("\nExamples missing XML:")
    display(missing[["case_id","dicom_name"]])

DICOM files: 410
XML files: 343
DICOMs with XML: 343
DICOMs missing XML: 67


,case_id,dicom_name,dicom_path,xml_path,has_xml
51,22579754,22579754_bbd6a3a35438c11b_MG_L_ML_ANON.dcm,../data/raw/inbreast/ALL-IMGS/22579754_bbd6a3a...,../data/raw/kaggle_inbreast/AllXML/22579754.xml,True
167,24058660,24058660_9e8db9e34d5275ef_MG_R_CC_ANON.dcm,../data/raw/inbreast/ALL-IMGS/24058660_9e8db9e...,../data/raw/kaggle_inbreast/AllXML/24058660.xml,True
386,53586415,53586415_dda3c6969a34ff8e_MG_L_CC_ANON.dcm,../data/raw/inbreast/ALL-IMGS/53586415_dda3c69...,NaN,False
168,24058686,24058686_9e8db9e34d5275ef_MG_L_CC_ANON.dcm,../data/raw/inbreast/ALL-IMGS/24058686_9e8db9e...,../data/raw/kaggle_inbreast/AllXML/24058686.xml,True
223,50993949,50993949_de5e8d61e501a71b_MG_L_CC_ANON.dcm,../data/raw/inbreast/ALL-IMGS/50993949_de5e8d6...,../data/raw/kaggle_inbreast/AllXML/50993949.xml,True
332,51049134,51049134_8c105bb715bf1c3c_MG_R_CC_ANON.dcm,../data/raw/inbreast/ALL-IMGS/51049134_8c105bb...,../data/raw/kaggle_inbreast/AllXML/51049134.xml,True
50,22579730,22579730_bbd6a3a35438c11b_MG_R_ML_ANON.dcm,../data/raw/inbreast/ALL-IMGS/22579730_bbd6a3a...,../data/raw/kaggle_inbreast/AllXML/22579730.xml,True
127,22678518,22678518_60995d51033e24b8_MG_L_ML_ANON.dcm,../data/raw/inbreast/ALL-IMGS/22678518_60995d5...,../data/raw/kaggle_inbreast/AllXML/22678518.xml,True
178,24065461,24065461_83db89f57aea498a_MG_R_CC_ANON.dcm,../data/raw/inbreast/ALL-IMGS/24065461_83db89f...,../data/raw/kaggle_inbreast/AllXML/24065461.xml,True
63,22580270,22580270_5530d5782fc89dd7_MG_L_ML_ANON.dcm,../data/raw/inbreast/ALL-IMGS/22580270_5530d57...,NaN,False



Examples missing XML:


,case_id,dicom_name
32,20588138,20588138_8d0b9620c53c0268_MG_R_ML_ANON.dcm
33,20588164,20588164_8d0b9620c53c0268_MG_R_CC_ANON.dcm
61,22580218,22580218_5530d5782fc89dd7_MG_L_CC_ANON.dcm
63,22580270,22580270_5530d5782fc89dd7_MG_L_ML_ANON.dcm
83,22613848,22613848_45c7f44839fd9e68_MG_L_ML_ANON.dcm
85,22613944,22613944_f23fa352e7de3dc7_MG_L_CC_ANON.dcm
87,22613996,22613996_f23fa352e7de3dc7_MG_L_ML_ANON.dcm
110,22670442,22670442_7e677f3d530e41ed_MG_R_CC_ANON.dcm
112,22670488,22670488_7e677f3d530e41ed_MG_R_ML_ANON.dcm
115,22670643,22670643_e15a16f87b4f9782_MG_L_CC_ANON.dcm


### 1.1 What the mapping audit shows

- **410** DICOMs in `ALL-IMGS/`, **343** XML annotation files.
- **343** DICOMs (every annotated case) have a matching XML — confirming the `file_id` join is 1-to-1 wherever it exists.
- **67** DICOMs have no XML at all. These are the "no annotation (Normal)" rows in `INbreast.xls`; they have BI-RADS 1 and are simply not annotated. The mass-extraction loop in Section 6 skips them by design (an XML is required to know where the mass is).

The non-trivial cases — XML files that exist but disagree with the XLS `Mass` column — are audited separately in `Step 0.3 — Investigate Annotation Sources`.


## 2. Setup — imports, paths, and output folders

Imports for the rest of the notebook plus the four key paths (`DATA_ROOT`, `DICOM_DIR`, `XML_DIR`, `XLS_PATH`) and the two output folders (`masses/benign/`, `masses/malignant/`). The cell creates the output folders if they don't exist and prints whether each input path actually resolves on disk — the existence checks are the first thing to look at if the notebook fails later.


In [2]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import pydicom
from tqdm import tqdm
from PIL import Image

from skimage.draw import polygon
from skimage.filters import threshold_otsu
from skimage.measure import label, regionprops
import plistlib

# =========================
# EDIT THESE PATHS
# =========================
DATA_ROOT = Path("../data/raw")   # <-- change
DICOM_DIR = DATA_ROOT / "inbreast/ALL-IMGS"                       # folder with .dcm files (or DICOM files without extension)
XML_DIR   = DATA_ROOT / "kaggle_inbreast" / "AllXML"        # folder with 12345678.xml files
XLS_PATH  = DATA_ROOT / "inbreast/INbreast.xls"                    # your excel file

# Output folder for the mass ROI crops + manifest.
# Structure: data/outputs/masses/{benign,malignant}/*.npy + mass_index.csv
OUT_ROOT = Path("../data/outputs") / "masses"
OUT_MASS_BENIGN    = OUT_ROOT / "benign"
OUT_MASS_MALIGNANT = OUT_ROOT / "malignant"

for p in [OUT_MASS_BENIGN, OUT_MASS_MALIGNANT]:
    p.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT:", DATA_ROOT)
print("DICOM_DIR exists?", DICOM_DIR.exists())
print("XML_DIR exists?", XML_DIR.exists())
print("XLS exists?", XLS_PATH.exists())
print("OUT_ROOT:", OUT_ROOT)

DATA_ROOT: ../data/raw
DICOM_DIR exists? True
XML_DIR exists? True
XLS exists? True
OUT_ROOT: ../data/outputs/masses


## 3. Build a `file_id` → DICOM path index

The XML files are named by `file_id` only (e.g. `20586908.xml`) while the DICOM filenames include the file_id plus the patient hash and view (`20586908_<hash>_MG_R_CC_ANON.dcm`). The next cell builds a `{int(file_id) → DICOM path}` dict so the extraction loop can look up the correct DICOM for each XML in O(1). The regex `^(\d+)_` selects the leading digits and ignores DICOMs that don't start with a numeric id.


In [3]:
def build_dicom_index(dicom_dir: Path):
    idx = {}
    for fp in dicom_dir.rglob("*"):
        if fp.is_dir():
            continue
        # many INbreast dicoms have no extension; accept everything that starts with digits
        m = re.match(r"^(\d+)_", fp.name)
        if m:
            idx[int(m.group(1))] = fp
    return idx

dicom_index = build_dicom_index(DICOM_DIR)
print("Indexed DICOMs:", len(dicom_index))
print("Example:", next(iter(dicom_index.items())) if dicom_index else None)

Indexed DICOMs: 410
Example: (22427705, PosixPath('../data/raw/inbreast/ALL-IMGS/22427705_d713ef5849f98b6c_MG_L_CC_ANON.dcm'))


## 4. Load `INbreast.xls` and derive benign / malignant labels from BI-RADS

This cell:

1. Opens `INbreast.xls`, finds the sheet containing `File Name` + `Bi-Rads` (Sheet1 — the per-image sheet — see `Step 0.2 — Map to XLS` for the audit), and reduces it to the columns we need (`File Name`, `Bi-Rads`, `ACR`, `View`, `Laterality`, `Acquisition date`).
2. Coerces `Bi-Rads` strings like `4a` / `4b` / `4c` to the integer 4 (`birads_int`), and applies the BI-RADS → mass-label mapping documented in the intro: 1 → ignore, 2–3 → benign, 4–6 → malignant. The mapped label lives in the `mass_label_from_birads` column.
3. Optionally locates the per-patient diagnosis sheet (Sheet2 with `Patient ID` + `DG`) if it's present — we don't use it for the mass label, but the audit step expects to see it.
4. Drops rows without a `file_id`, deduplicates on `file_id`, and re-indexes by it so the extraction loop in Section 6 can do `image_meta.loc[file_id, "mass_label_from_birads"]`.


In [4]:
xls = pd.ExcelFile(XLS_PATH)
print("Sheets:", xls.sheet_names)

sheets = {name: xls.parse(name) for name in xls.sheet_names}

def find_sheet_with_cols(required_cols):
    for name, df in sheets.items():
        cols = {c.strip().lower() for c in df.columns.astype(str)}
        if all(rc.lower() in cols for rc in required_cols):
            return name, df
    return None, None

main_name, main_df = find_sheet_with_cols(["File Name", "Bi-Rads"])
if main_df is None:
    raise ValueError("Couldn't find the main sheet with columns: File Name, Bi-Rads")

print("Main sheet:", main_name)
main_df = main_df.rename(columns={c: c.strip() for c in main_df.columns})

# keep only what we need (add ACR if you want)
keep_cols = [c for c in ["File Name", "Bi-Rads", "ACR", "View", "Laterality", "Acquisition date"] if c in main_df.columns]
main = main_df[keep_cols].copy()
main["file_id"] = pd.to_numeric(main["File Name"], errors="coerce").astype("Int64")

# optional diagnosis sheet (DG)
diag_name, diag_df = find_sheet_with_cols(["Patient ID", "DG"])
diag = None
if diag_df is not None:
    diag_df = diag_df.rename(columns={c: c.strip() for c in diag_df.columns})
    diag = diag_df.copy()
    diag["patient_id"] = pd.to_numeric(diag["Patient ID"], errors="coerce").astype("Int64")
    print("Diagnosis sheet:", diag_name, "rows:", len(diag))
else:
    print("No DG diagnosis sheet detected (that's okay).")

def birads_to_int(b):
    """Converts 4a/4b/4c → 4, etc."""
    if pd.isna(b):
        return None
    s = str(b).strip().lower()
    m = re.match(r"^(\d+)", s)
    return int(m.group(1)) if m else None

def birads_to_mass_label(b):
    """
    WARNING:
    BI-RADS is suspicion, not pathology ground truth.
    This mapping is only to satisfy your requested rule.
    """
    bi = birads_to_int(b)
    if bi is None:
        return None
    if bi in [1]:
        return None  # normal / ignore for mass classification
    if bi in [2, 3]:
        return "benign"
    if bi in [4, 5, 6]:
        return "malignant"
    return None

main["birads_int"] = main["Bi-Rads"].apply(birads_to_int)
main["mass_label_from_birads"] = main["Bi-Rads"].apply(birads_to_mass_label)

# Per-image lookup:
image_meta = main.dropna(subset=["file_id"]).drop_duplicates(subset=["file_id"]).set_index("file_id")
print("Unique images with metadata:", len(image_meta))
image_meta.head()

Sheets: ['Sheet1', 'Sheet2']
Main sheet: Sheet1
Diagnosis sheet: Sheet2 rows: 412
Unique images with metadata: 410


,File Name,Bi-Rads,ACR,View,Laterality,Acquisition date,birads_int,mass_label_from_birads
file_id,,,,,,,,
22678622,22678622.0,1,4,CC,R,201001.0,1.0,NaN
22678646,22678646.0,3,4,CC,L,201001.0,3.0,benign
22678670,22678670.0,1,4,MLO,R,201001.0,1.0,NaN
22678694,22678694.0,3,4,MLO,L,201001.0,3.0,benign
22614074,22614074.0,5,2,CC,R,201001.0,5.0,malignant


## 5. XML parser — INbreast XMLs are OsiriX plists, not generic XML

Many INbreast `.xml` files are actually Apple plist format (OsiriX export), not the kind of XML you parse with `xml.etree`. The cleanest way to read them is with the standard-library `plistlib`. The next cell defines:

- `_parse_point_any(p)` / `_coerce_points(pts)` — tolerant point-list coercion. Different INbreast XMLs store the polygon vertices in different shapes (lists of `(x, y)` strings, flat number lists, nested lists), so the parser accepts all the common forms.
- `load_rois_from_inbreast_xml(xml_path)` — returns a list of `{"name": <ROI name>, "points": Nx2 (x, y) float array}` dicts. Only ROIs with at least 3 polygon vertices are returned (a polygon with < 3 points can't be rasterised).

Section 6 then filters this list to ROIs with `name.lower() == "mass"`.


In [5]:
import re
import numpy as np
import plistlib

_num_re = re.compile(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?")

def _parse_point_any(p):
    """
    Accepts:
      - (x, y) tuple/list
      - [x, y] list
      - "(x, y)" string
      - "x, y" string
    Returns (x, y) floats or None.
    """
    # already numeric pair
    if isinstance(p, (list, tuple)) and len(p) == 2:
        try:
            return float(p[0]), float(p[1])
        except Exception:
            return None

    # string "(x, y)" or "x, y"
    if isinstance(p, str):
        nums = _num_re.findall(p)
        if len(nums) >= 2:
            return float(nums[0]), float(nums[1])
        return None

    return None

def _coerce_points(pts):
    """
    Handles:
      - pts = [[x,y], [x,y], ...]
      - pts = ["(x,y)", "(x,y)", ...]
      - pts = "(x,y) (x,y) ..."  (rare)
    Returns Nx2 float array (x,y). Empty array if can't parse.
    """
    out = []

    if pts is None:
        return np.zeros((0, 2), dtype=float)

    # If it's a single string that may contain multiple points
    if isinstance(pts, str):
        nums = _num_re.findall(pts)
        # interpret as x1,y1,x2,y2,...
        for i in range(0, len(nums) - 1, 2):
            out.append((float(nums[i]), float(nums[i+1])))
        return np.array(out, dtype=float) if out else np.zeros((0, 2), dtype=float)

    # If it's a list/tuple of points (strings or numeric pairs)
    if isinstance(pts, (list, tuple)):
        for p in pts:
            xy = _parse_point_any(p)
            if xy is not None:
                out.append(xy)
            else:
                # sometimes it's nested weirdly; try flattening
                if isinstance(p, (list, tuple)) and len(p) > 2:
                    # try reading as consecutive pairs
                    try:
                        flat = list(p)
                        for i in range(0, len(flat) - 1, 2):
                            out.append((float(flat[i]), float(flat[i+1])))
                    except Exception:
                        pass

    return np.array(out, dtype=float) if out else np.zeros((0, 2), dtype=float)

def load_rois_from_inbreast_xml(xml_path):
    """
    Returns list of ROI dicts with:
      - name: ROI name (e.g., 'Mass', 'Calcification')
      - points: Nx2 array of (x, y) pixel points
    """
    with open(xml_path, "rb") as f:
        data = plistlib.load(f)

    img0 = data["Images"][0]
    rois = img0.get("ROIs", [])

    out = []
    for roi in rois:
        name = str(roi.get("Name", "")).strip()

        # OsiriX exports usually have Point_px, but keep fallback if needed
        pts_raw = roi.get("Point_px", None)
        if pts_raw is None:
            pts_raw = roi.get("Points", None)

        pts = _coerce_points(pts_raw)

        # need at least 3 points to form a polygon
        if pts.shape[0] < 3:
            continue

        out.append({"name": name, "points": pts})

    return out

## 6. Extract mass crops + save to `masses/`

The main loop. For each XML file:

1. Look up the DICOM by `file_id` (Section 3's index) and the BI-RADS label by `file_id` (Section 4's `image_meta`).
2. Skip if the label maps to "ignore" (BI-RADS 1) — those images are normal.
3. Parse the XML (Section 5) and keep only ROIs with `name == "Mass"`. Skip the file if there are none (some annotated cases have only calcifications, distortions, or asymmetries).
4. For each remaining Mass ROI: rasterise its polygon to a binary mask, compute the tight bounding box with a 20-pixel margin, crop the raw DICOM pixel array, and save the crop as `<file_id>_mass<i>.npy` plus an 8-bit `.png` preview into the appropriate `benign/` or `malignant/` folder.
5. Append a row to the manifest with the bounding-box coordinates and source paths.

This produces 116 mass crops total — Section 7 below compares this number to the reference paper's 112.


In [6]:
def read_dicom_as_float(path):
    """Read a DICOM file and return pixel data as float32.

    No VOI LUT application and no MONOCHROME1 inversion: all 410
    INbreast files are MONOCHROME2 with no VOI LUT metadata in the
    headers (audited in Step 0.1), so both branches were no-ops on
    this dataset and have been removed.
    """
    ds = pydicom.dcmread(str(path))
    img = ds.pixel_array.astype(np.float32)
    return img

def polygon_to_mask(points, shape_hw):
    """points: Nx2 array of (x, y), shape_hw: (H, W)."""
    pts = np.asarray(points, dtype=float)
    if pts.shape[0] < 3:
        return np.zeros(shape_hw, dtype=bool)
    rr, cc = polygon(pts[:, 1], pts[:, 0], shape_hw)
    mask = np.zeros(shape_hw, dtype=bool)
    mask[rr, cc] = True
    return mask

def save_patch_npy_and_png(patch, out_stem: Path):
    np.save(str(out_stem.with_suffix(".npy")), patch.astype(np.float32))
    # PNG preview (normalized)
    p = patch.astype(np.float32)
    p = p - np.nanmin(p)
    if np.nanmax(p) > 0:
        p = p / np.nanmax(p)
    img8 = (np.clip(p, 0, 1) * 255).astype(np.uint8)
    Image.fromarray(img8).save(str(out_stem.with_suffix(".png")))

def crop_from_mask(image, mask, margin=20):
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return None
    y0, y1 = ys.min(), ys.max()
    x0, x1 = xs.min(), xs.max()
    H, W = image.shape
    y0 = max(0, y0 - margin); y1 = min(H - 1, y1 + margin)
    x0 = max(0, x0 - margin); x1 = min(W - 1, x1 + margin)
    return image[y0:y1+1, x0:x1+1], (y0, y1, x0, x1)

manifest_rows = []

xml_files = sorted(XML_DIR.glob("*.xml"))
print("XML files:", len(xml_files))

for xml_path in tqdm(xml_files):
    file_id = int(xml_path.stem) if xml_path.stem.isdigit() else None
    if file_id is None:
        continue
    if file_id not in dicom_index:
        continue
    if file_id not in image_meta.index:
        continue

    # label from BI-RADS (as requested)
    label_from_birads = image_meta.loc[file_id, "mass_label_from_birads"]
    if label_from_birads not in ["benign", "malignant"]:
        # skip normals/unknowns
        continue

    img = read_dicom_as_float(dicom_index[file_id])
    H, W = img.shape

    rois = load_rois_from_inbreast_xml(xml_path)
    mass_rois = [r for r in rois if r["name"].strip().lower() == "mass"]
    if not mass_rois:
        continue

    out_dir = OUT_MASS_BENIGN if label_from_birads == "benign" else OUT_MASS_MALIGNANT

    for j, roi in enumerate(mass_rois):
        mask = polygon_to_mask(roi["points"], (H, W))
        cropped = crop_from_mask(img, mask, margin=20)
        if cropped is None:
            continue
        patch, (y0, y1, x0, x1) = cropped

        out_stem = out_dir / f"{file_id}_mass{j:02d}"
        save_patch_npy_and_png(patch, out_stem)

        manifest_rows.append({
            "file_id": file_id,
            "roi_index": j,
            "label": label_from_birads,
            "y0": y0, "y1": y1, "x0": x0, "x1": x1,
            "dicom_path": str(dicom_index[file_id]),
            "xml_path": str(xml_path),
            "birads": image_meta.loc[file_id, "Bi-Rads"] if "Bi-Rads" in image_meta.columns else None
        })

manifest = pd.DataFrame(manifest_rows)
manifest_path = OUT_ROOT / "mass_index.csv"
manifest.to_csv(manifest_path, index=False)

print("Saved mass crops:", len(manifest))
print("Manifest:", manifest_path)
manifest.head()

XML files: 343


100%|██████████| 343/343 [01:00<00:00,  5.68it/s]

Saved mass crops: 116
Manifest: ../data/outputs/masses/mass_index.csv


,file_id,roi_index,label,y0,y1,x0,x1,dicom_path,xml_path,birads
0,20586908,0,benign,963,1117,2373,2511,../data/raw/inbreast/ALL-IMGS/20586908_6c613a1...,../data/raw/kaggle_inbreast/AllXML/20586908.xml,2
1,20586908,1,benign,988,1221,3091,3327,../data/raw/inbreast/ALL-IMGS/20586908_6c613a1...,../data/raw/kaggle_inbreast/AllXML/20586908.xml,2
2,20586934,0,malignant,2021,2222,105,331,../data/raw/inbreast/ALL-IMGS/20586934_6c613a1...,../data/raw/kaggle_inbreast/AllXML/20586934.xml,5
3,20586960,0,benign,1264,1416,1779,1909,../data/raw/inbreast/ALL-IMGS/20586960_6c613a1...,../data/raw/kaggle_inbreast/AllXML/20586960.xml,2
4,20586960,1,benign,794,985,2445,2720,../data/raw/inbreast/ALL-IMGS/20586960_6c613a1...,../data/raw/kaggle_inbreast/AllXML/20586960.xml,2


### 6.1 What was extracted

- **116 mass `.npy` crops** total — split into **75 malignant + 41 benign**.
- Spread across **107 distinct images** (most images have one Mass ROI; 8 images contain multiple — 7 with 2 each, 1 with 3 — giving 116 ROIs from 107 images).
- The 116 saved crops are at their **native DICOM resolution** (variable, ~200–700 px per side). Each one's bounding-box coordinates are recorded in `mass_index.csv` so Steps 2.1 / 2.2 / 2.4 can re-crop at 224×224 from the preprocessed image rather than re-cropping from the raw DICOM.

The 116-vs-paper-112 reconciliation is in Section 7.


## 7. Comparison with the reference paper

This notebook extracts **116 mass patches** (75 malignant + 41 benign), from **107 distinct images** (the 8 images that contain more than one Mass ROI contribute 9 extra patches above one-per-image). The reference paper reports **112 mass patches** (76 malignant + 36 benign).

The 116-vs-112 gap is the combination of two effects (fuller breakdown in Section 7.1 below):

1. **Multi-ROI retention.** We keep every `Name="Mass"` ROI from the XML. Collapsing to one ROI per image would give 107 patches (35 benign + 72 malignant; see §7.1).
2. **"Mass tissue" inclusion criterion.** The paper appears to additionally fold in a handful of Asymmetry- or Distortion-only XMLs as "mass tissue" — we do not. Adding 1 BI-RADS-2 distortion + 4 BI-RADS-4/5 asymmetry / distortion images to the 107 baseline (35 + 72) gives 36 + 76 = 112 — the paper's number exactly.

Our criterion is exactly reproducible from the public INbreast XML alone (`Name == "Mass"`); the paper's selection requires per-image judgements about which non-Mass findings to include, which are not documented.

The downstream consumers also apply a defensive filter that drops one entirely-zero patch (`file_id 50994354`, BI-RADS 4a, view `MG_R_ML_ANON`), reducing the usable mass dataset to **115 patches (74 malignant + 41 benign)**. That filter lives at the downstream consumers (Steps 2.1 / 2.2 / 2.3 / 2.4), not here — this notebook saves all 116 patches; the filter is applied at load time (root cause and effect in Section 7.1 below).

### 7.1 Full detail — multi-ROI breakdown and the data-quality filter

*Recorded here in full so this detail is preserved independently of the report (all figures verified against `mass_index.csv`).*

**Which images are multi-ROI.** The 8 images with more than one `Name="Mass"` ROI split as **5 BI-RADS-2 benign** (`20586908`, `20586960`, `22580341`, `22580367`, and `51049107` — the last carrying **3** ROIs), contributing **6** ROIs above one-per-image, and **3 BI-RADS-5 malignant** (`22427840`, `22579730`, `22613770`), contributing **3** extra — 9 extra ROIs in total (116 = 107 + 9). These are a mix of two phenomena: **near-duplicate** annotations of a single mass (e.g. `51049107`'s three closely-overlapping benign outlines) and **genuinely multifocal** disease (e.g. `22427840`, BI-RADS 5, two well-separated malignant foci — multifocality is well documented in high-BI-RADS cases). Exact ROI boxes are in `mass_index.csv`. Retaining all ROIs preserves the genuine multifocal masses; collapsing to one-per-image would give 107 patches (35 benign + 72 malignant).

**"Mass tissue" inclusion criterion.** We extract only ROIs literally tagged `Name="Mass"`. INbreast additionally holds images whose XML carries only `Asymmetry` or `Distortion` ROIs and no `Mass` ROI; both the XML `Name` and the XLS `Mass` column agree these are not masses, yet several carry BI-RADS 4a–5, so radiologically they may behave like masses. The paper appears to fold a few of these in as "mass tissue" (the 1-distortion + 4-asymmetry/distortion additions in the Section 7 reconstruction come from this pool); we do not.

**Data-quality filter — root cause and effect.** The single all-zero patch is `file_id 50994354` (BI-RADS 4a, malignant, view `MG_R_ML_ANON`). Its Mass box is `y ∈ [806, 1098]`, `x ∈ [2214, 2483]` on the 3328×2560 preprocessed image — a 292×269 = **78,548-pixel** box lying **entirely outside** the segmented breast (the crop comes back uniformly zero, i.e. none of its pixels are inside the breast mask, even though the breast region itself is intact). The cause is an annotation/orientation mismatch: for a **right-laterality MLO** view the breast occupies the **left** of the image, so a box in the far-right x-range sits on the empty chest-wall side. Step 1.1 zeros all non-breast pixels before saving, so cropping this box yields a uniformly-zero 224×224 patch. The dynamic downstream filter (`np.load(p).max() > 0`) drops it, reducing the mass set from 116 (75+41) to **115 (74+41)**. Removing this guaranteed-wrong prediction slightly raises each method's held-out test accuracy while leaving CV mean/std and the method ranking unchanged; no other INbreast mass annotation shows this pattern.

## 8. Summary

What this notebook produces:

- **116 mass `.npy` crops** under `data/outputs/masses/{benign,malignant}/`, named `<file_id>_mass<i>.npy`.
- Matching 8-bit `.png` previews (per-patch min-max — for visual inspection only).
- **`mass_index.csv`** with one row per ROI: `file_id`, `roi_index`, `label`, bounding box `(y0, y1, x0, x1)`, `dicom_path`, `xml_path`, `birads`.

What this notebook does **not** do:

- It does not preprocess the DICOM (no percentile clip, no breast mask, no CLAHE) — the saved `.npy` is the raw float pixel array cropped to the bbox. Downstream stages re-crop from the **preprocessed** full image (`data/outputs/preprocessed/final/*.npy`) at 224×224 using the bbox coordinates from this manifest.
- It does not apply the all-zero-patch data-quality filter — that lives at the downstream consumers (Section 7).
- It does not extract background (fatty / fibroglandular) patches — that is Step 1.3.

Audit trail:

- Counts per BI-RADS, mass-label, and per-file are visible in the printed outputs of Cells in Sections 4 and 6.
- The full audit of the inter-annotator inconsistency between the XLS `Mass` column (108 files flagged) and the XML `Mass` ROIs (107 files) is in **Step 0.3 — Investigate Annotation Sources**.
- The 116-vs-112 paper reconciliation is in Sections 7 and 7.1 above.
